# 3. Data Preparation: Pembersihan Data dan Rekayasa 12 Fitur Spektral Terarah

Tahap **Data Preparation** dalam metodologi **CRISP-DM** bertugas mentransformasikan data mentah reflektansi Sentinel-2 menjadi representasi fitur bernilai tinggi yang siap diproses oleh algoritma pembelajaran mesin.

### Arsitektur Data Preparation Berbasis Separasi Dua Arah:
1. **Pembersihan Data (*Data Cleaning & QA*)**:
   - Menghapus nilai pantulan jenuh awan ($> 1.0$) atau nilai negatif ($< 0.0$).
   - Mengeliminasi sampel anomali kontradiktif (misal: piksel perairan dengan pantulan NIR/NDVI tinggi vegetatif).
2. **Rekayasa 12 Fitur Prediktor Inti (*Feature Engineering*)**:
   - **7 Band Reflektansi Asli BOA Sentinel-2**: `B02`, `B03`, `B04`, `B08`, `B8A`, `B11`, `B12`.
   - **5 Indeks Biofisik Terarah Fiks**:
     - `REMI`: Memisahkan Hutan Mangrove secara tegas dari Hutan Daratan.
     - `NDVI`: Mengukur kerapatan klorofil kanopi darat & bermitra dengan REMI.
     - `IBI`: Mengisolasi material Bangunan dan meredam tanah kosong/sawah kering.
     - `MNDWI`: Menangkap kelengasan tanah sawah berair/kosongan vs air murni.
     - `NDWI`: Menarik garis batas perairan terbuka murni secara tegas.
3. **Penyelarasan Skala & Pembagian Data (Stratified 80:20 Split)**:
   - Membagi data latih (*training set*) dan data uji (*testing set*) secara seimbang dengan pembakuan skala `StandardScaler`.


In [1]:
# impor library persiapan data
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# atur gaya visualisasi
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('[OK] Seluruh library Data Preparation berhasil dimuat.')


[OK] Seluruh library Data Preparation berhasil dimuat.


## 3.1 Pemuatan Data dan Pembersihan Kualitas Sampel Poligon (*Polygon Cleaning*)

Memuat dataset poligon dari `2-data_understanding/extracted_data/sentinel2_sample_polygons.csv` dan memfilter anomali spektral kontradiktif.


In [2]:
# muat dataset poligon mentah
data_dir = os.path.join('..', '2-data_understanding', 'extracted_data')
raw_path = os.path.join(data_dir, 'sentinel2_sample_polygons.csv')
df_raw = pd.read_csv(raw_path)

print(f'Dimensi Dataset Poligon Mentah: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom')
print(f'Total Nilai Hilang (NaN): {df_raw.isnull().sum().sum()}')
print(f'Total Nilai Tak Terhingga (Inf): {np.isinf(df_raw.select_dtypes(include=np.number)).values.sum()}')
print(f'Total Data Duplikat: {df_raw.duplicated().sum()}')

# prosedur pembersihan anomali fisis poligon
def clean_poly_func(r):
    bands = [r['B02'], r['B03'], r['B04'], r['B08'], r['B8A'], r['B11'], r['B12']]
    if any(b <= 0.0 or b >= 1.0 for b in bands): return False
    c = r['class_name']
    ndvi = r['NDVI']
    if c == 'perairan' and (ndvi > 0.15 or r['B08'] > 0.16): return False
    if c == 'bangunan' and ndvi > 0.35: return False
    if c == 'hutan' and ndvi < 0.35: return False
    if c == 'mangrove' and ndvi < 0.35: return False
    return True

df = df_raw[df_raw.apply(clean_poly_func, axis=1)].copy().reset_index(drop=True)
class_map = {'bangunan': 0, 'pertanian': 1, 'perairan': 2, 'hutan': 3, 'mangrove': 4}
df['label'] = df['class_name'].map(class_map)

# tampilkan sebaran poligon bersih
print(f'\nDimensi Setelah Pembersihan: {df.shape[0]} poligon valid')
print('\nDistribusi Poligon Bersih per Kelas Tutupan Lahan:')
display(df['class_name'].value_counts().reset_index())


Dimensi Dataset Poligon Mentah: 250 baris, 18 kolom
Total Nilai Hilang (NaN): 0
Total Nilai Tak Terhingga (Inf): 0
Total Data Duplikat: 0

Dimensi Setelah Pembersihan: 218 poligon valid

Distribusi Poligon Bersih per Kelas Tutupan Lahan:


,class_name,count
0,pertanian,50
1,bangunan,47
2,mangrove,43
3,perairan,43
4,hutan,35


## 3.2 Rekayasa 12 Fitur Spektral Inti (*Feature Engineering*)

Membentuk **12 Fitur Prediktor Inti (7 Band Reflektansi + 5 Indeks Biofisik Terarah)**:
1. **`B02`, `B03`, `B04`, `B08`, `B8A`, `B11`, `B12`**: Reflektansi BOA 7 band dasar Sentinel-2.
2. **`REMI`**: $\frac{B8A - B04}{(B11 - B03) + \epsilon}$ (Isolasi Mangrove vs Hutan Daratan).
3. **`NDVI`**: $\frac{B08 - B04}{B08 + B04 + \epsilon}$ (Biomassa Kanopi Klorofil Rapat).
4. **`IBI`**: $\frac{NDBI - (SAVI + MNDWI)/2}{NDBI + (SAVI + MNDWI)/2 + \epsilon}$ (Isolasi Bangunan & Peredam Tanah Kosong/Sawah Kering).
5. **`MNDWI`**: $\frac{B03 - B11}{B03 + B11 + \epsilon}$ (Deteksi Sawah Berair & Kelembapan Substrat).
6. **`NDWI`**: $\frac{B03 - B08}{B03 + B08 + \epsilon}$ (Deliniasi Garis Batas Air Murni).


In [3]:
# hitung 5 indeks biofisik terarah pada poligon
eps = 1e-6
b02, b03, b04 = df['B02'].values, df['B03'].values, df['B04'].values
b08, b8a = df['B08'].values, df['B8A'].values
b11, b12 = df['B11'].values, df['B12'].values

df['REMI'] = (b8a - b04) / ((b11 - b03) + eps)
df['NDVI'] = (b08 - b04) / (b08 + b04 + eps)

ndbi = (b11 - b08) / (b11 + b08 + eps)
savi = ((b08 - b04) * 1.5) / (b08 + b04 + 0.5 + eps)
mndwi = (b03 - b11) / (b03 + b11 + eps)
df['IBI'] = (ndbi - (savi + mndwi) / 2.0) / (ndbi + (savi + mndwi) / 2.0 + eps)
df['MNDWI'] = mndwi
df['NDWI'] = (b03 - b08) / (b03 + b08 + eps)

# tetapkan 12 fitur prediktor inti
feature_cols = [
    'B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12',
    'REMI', 'NDVI', 'IBI', 'MNDWI', 'NDWI'
]

print(f'[OK] Berhasil merekayasa {len(feature_cols)} Fitur Inti (7 Band Reflektansi + 5 Indeks Biofisik Terarah):')
for i, fcol in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {fcol}')

# tampilkan ringkasan rata-rata 12 fitur per kelas
display(df.groupby('class_name')[feature_cols].mean().round(4))


[OK] Berhasil merekayasa 12 Fitur Inti (7 Band Reflektansi + 5 Indeks Biofisik Terarah):
   1. B02
   2. B03
   3. B04
   4. B08
   5. B8A
   6. B11
   7. B12
   8. REMI
   9. NDVI
  10. IBI
  11. MNDWI
  12. NDWI


,B02,B03,B04,B08,B8A,B11,B12,REMI,NDVI,IBI,MNDWI,NDWI
class_name,,,,,,,,,,,,
bangunan,0.1709,0.1887,0.2123,0.2473,0.2569,0.3116,0.2921,0.4374,0.0991,2.4974,-0.2853,-0.1750
hutan,0.0472,0.0673,0.0567,0.2724,0.2884,0.1842,0.1075,4.9606,0.6719,0.6371,-0.4717,-0.6150
mangrove,0.0400,0.0656,0.0425,0.2931,0.3064,0.1336,0.0636,3.9447,0.7484,1.3500,-0.3268,-0.6350
perairan,0.1028,0.1091,0.0926,0.0776,0.0826,0.0720,0.0589,1.2930,-0.0827,-0.8787,0.2210,0.1769
pertanian,0.0770,0.1036,0.1132,0.2436,0.2598,0.2339,0.1658,0.7872,0.3802,5.7849,-0.3692,-0.4016


## 3.3 Pembagian Data Poligon (Stratified 80:20 Split) dan Penskalaan Fitur

Membagi dataset poligon dengan stratifikasi kelas target dan melakukan standardisasi fitur menggunakan `StandardScaler`.


In [4]:
# pembagian data poligon stratified 80:20
train_df, test_df = train_test_split(df, test_size=0.20, random_state=2024, stratify=df['label'])

prep_dir = 'prepared_data'
os.makedirs(prep_dir, exist_ok=True)

# simpan dataset poligon bersih
df.to_csv(os.path.join(prep_dir, 'data_prepared_all.csv'), index=False)
train_df.to_csv(os.path.join(prep_dir, 'train_data.csv'), index=False)
test_df.to_csv(os.path.join(prep_dir, 'test_data.csv'), index=False)

# sinkronisasi ke direktori deployment
deploy_prep_dir = os.path.join('..', '6-deployment', 'prepared_data')
if os.path.exists(deploy_prep_dir):
    df.to_csv(os.path.join(deploy_prep_dir, 'data_prepared_all.csv'), index=False)

# fit standard scaler pada data latih poligon
scaler = StandardScaler()
scaler.fit(train_df[feature_cols])
joblib.dump(scaler, os.path.join(prep_dir, 'scaler.pkl'))

print(f'[OK] Dataset Poligon Berhasil Disimpan:')
print(f'  - Total Data Poligon : {len(df)} sampel')
print(f'  - Data Poligon Latih : {len(train_df)} sampel')
print(f'  - Data Poligon Uji   : {len(test_df)} sampel')


[OK] Dataset Poligon Berhasil Disimpan:
  - Total Data Poligon : 218 sampel
  - Data Poligon Latih : 174 sampel
  - Data Poligon Uji   : 44 sampel


## 3.4 Pembersihan dan Rekayasa Dataset Spektral Piksel (*Pixel-Level Dataset Preparation*)

Dataset piksel (12.500 sampel) merupakan **basis data operasional utama model klasifikasi**. Menerapkan filter pembersihan fisis spektral dan kalkulasi 5 indeks terarah yang identik.


In [5]:
# muat dataset piksel mentah
pixels_raw_path = os.path.join(data_dir, 'sentinel2_pixels_dataset.csv')
df_pixels_raw = pd.read_csv(pixels_raw_path)
print(f'Dimensi Dataset Piksel Awal: {df_pixels_raw.shape[0]} baris, {df_pixels_raw.shape[1]} kolom')

# prosedur pembersihan anomali fisis piksel
def clean_pixel(r):
    bands = [r['B02'], r['B03'], r['B04'], r['B08'], r['B8A'], r['B11'], r['B12']]
    if any(b <= 0.0 or b >= 1.0 for b in bands):
        return False
    c = r['class_name']
    ndvi = r['NDVI']
    b8 = r['B08']
    if c == 'perairan':
        if ndvi > 0.12 or b8 > 0.14: return False
    elif c == 'bangunan':
        if ndvi > 0.35: return False
    elif c == 'hutan':
        if ndvi < 0.35: return False
    elif c == 'mangrove':
        if ndvi < 0.35: return False
    return True

clean_pixels = df_pixels_raw[df_pixels_raw.apply(clean_pixel, axis=1)].copy().reset_index(drop=True)

# kalkulasi 5 indeks biofisik terarah pada piksel
eps = 1e-6
pb02, pb03, pb04 = clean_pixels['B02'].values, clean_pixels['B03'].values, clean_pixels['B04'].values
pb08, pb8a = clean_pixels['B08'].values, clean_pixels['B8A'].values
pb11, pb12 = clean_pixels['B11'].values, clean_pixels['B12'].values

clean_pixels['REMI'] = (pb8a - pb04) / ((pb11 - pb03) + eps)
clean_pixels['NDVI'] = (pb08 - pb04) / (pb08 + pb04 + eps)

p_ndbi = (pb11 - pb08) / (pb11 + pb08 + eps)
p_savi = ((pb08 - pb04) * 1.5) / (pb08 + pb04 + 0.5 + eps)
p_mndwi = (pb03 - pb11) / (pb03 + pb11 + eps)
clean_pixels['IBI'] = (p_ndbi - (p_savi + p_mndwi) / 2.0) / (p_ndbi + (p_savi + p_mndwi) / 2.0 + eps)
clean_pixels['MNDWI'] = p_mndwi
clean_pixels['NDWI'] = (pb03 - pb08) / (pb03 + pb08 + eps)
clean_pixels['label'] = clean_pixels['class_name'].map(class_map)

print(f'Dimensi Dataset Piksel Setelah Pembersihan: {clean_pixels.shape[0]} piksel valid')

# pembagian data piksel stratified 80:20
train_pix, test_pix = train_test_split(clean_pixels, test_size=0.20, random_state=2024, stratify=clean_pixels['label'])

# simpan dataset piksel bersih ke prepared_data
clean_pixels.to_csv(os.path.join(prep_dir, 'pixels_prepared_all.csv'), index=False)
train_pix.to_csv(os.path.join(prep_dir, 'train_pixels.csv'), index=False)
test_pix.to_csv(os.path.join(prep_dir, 'test_pixels.csv'), index=False)

# fit standard scaler pada data latih piksel
scaler_pix = StandardScaler()
scaler_pix.fit(train_pix[feature_cols])
joblib.dump(scaler_pix, os.path.join(prep_dir, 'scaler_pixels.pkl'))

# sinkronisasi dataset piksel ke direktori deployment
if os.path.exists(deploy_prep_dir):
    clean_pixels.to_csv(os.path.join(deploy_prep_dir, 'pixels_prepared_all.csv'), index=False)
    train_pix.to_csv(os.path.join(deploy_prep_dir, 'train_pixels.csv'), index=False)
    test_pix.to_csv(os.path.join(deploy_prep_dir, 'test_pixels.csv'), index=False)
    joblib.dump(scaler_pix, os.path.join(deploy_prep_dir, 'scaler_pixels.pkl'))

print(f'[OK] Dataset Piksel Berhasil Disimpan:')
print(f'  - Total Piksel Bersih : {len(clean_pixels)} sampel')
print(f'  - Data Piksel Latih   : {len(train_pix)} sampel')
print(f'  - Data Piksel Uji     : {len(test_pix)} sampel')
print('\nDistribusi Piksel per Kelas Tutupan Lahan:')
display(clean_pixels['class_name'].value_counts().reset_index())


Dimensi Dataset Piksel Awal: 12500 baris, 15 kolom
Dimensi Dataset Piksel Setelah Pembersihan: 10537 piksel valid
[OK] Dataset Piksel Berhasil Disimpan:
  - Total Piksel Bersih : 10537 sampel
  - Data Piksel Latih   : 8429 sampel
  - Data Piksel Uji     : 2108 sampel

Distribusi Piksel per Kelas Tutupan Lahan:


,class_name,count
0,pertanian,2500
1,bangunan,2256
2,mangrove,2184
3,perairan,1903
4,hutan,1694
